# Libraries

In [ ]:
import os
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\moous\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\moous\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# loading and preprocessing data

In [ ]:
import re
import os
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

base_path = "TP/All-in-many/"
label_path = "TP/All-in-many_classification/"


def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return tokens

def extract_article_number(filename):
    # Extract the number after "Article_" and before "_Volume"
    match = re.search(r'Article_(\d+)_Volume', filename)
    if match:
        return int(match.group(1))
    else:
        return -1  # fallback if no number found

# Get sorted list of article files
sorted_files = sorted(os.listdir(base_path), key=extract_article_number)

data = []
labels = []

for filename in sorted_files:
    # Skip hidden files
    if filename.startswith('.'):
        continue
    
    article_file = os.path.join(base_path, filename)
    label_file = os.path.join(label_path, filename)
    
    if os.path.isfile(article_file) and os.path.isfile(label_file):
        # Read article
        try:
            with open(article_file, encoding='utf-8') as f:
                text = f.read()
        except UnicodeDecodeError:
            with open(article_file, encoding='latin-1') as f:
                text = f.read()
        
        # Read label
        try:
            with open(label_file, encoding='utf-8') as f:
                label = f.read().strip()
        except UnicodeDecodeError:
            with open(label_file, encoding='latin-1') as f:
                label = f.read().strip()
        
        data.append(preprocess_text(text))
        labels.append(label)
    else:
        print(f"⚠️ Missing match for {filename}")

print(f"\n✅ Loaded {len(data)} documents in correct order")
print("Classes found:", set(labels))



✅ Loaded 117 documents in correct order
Classes found: {'1', '3', '4', '2'}


# Probabilities calculations

for each document we need to compute the probability of each class

The most probable class 𝒄" for a given document 𝒅 is computed by selecting the class with the highest product of two probabilities:

o The prior probability of the class 𝑷(𝒄)

o The probability of the document given the class 𝑷(𝒅|𝒄)

              𝒄 = 𝐚𝐫𝐠 𝐦𝐚𝐱 𝑷(𝒄) 𝑷(𝒅|𝒄)
  
o The document 𝒅 can be represented as a set of words:
              𝒄 = 𝐚𝐫𝐠 𝐦𝐚𝐱 𝑷(𝒄) 𝑷(𝒘𝟏, 𝒘𝟐, 𝒘𝟑 … , 𝒘𝒏|𝒄)

o The probability 𝑷(𝒄) represents the proportion of documents in the training set that belong to class c
              
              𝑷(𝒄) = 𝑵𝒄 / 𝑵


𝑵𝒄 is the number of documents in the training set belonging to class 𝒄.

𝑵 is the total number of documents in the training set.

o The probability 𝑷(𝒘𝟏, 𝒘𝟐, 𝒘𝟑 … , 𝒘𝒏|𝒄) is computed as the product of the individual probabilities:

            𝑷(𝒘𝟏, 𝒘𝟐, 𝒘𝟑 … , 𝒘𝒏|𝒄) = product 𝑷(𝒘𝒊|𝒄) i ranging from 1 to n

Where:

The probability 𝑷(𝒘𝒊|𝒄) corresponds to the relative frequency of the word 𝒘𝒊 among all words in documents belonging to class 𝒄:

              𝑷(𝒘𝒊|𝒄) = (𝐜𝐨𝐮𝐧𝐭(𝒘𝒊, 𝒄) + 𝟏) / ∑ 𝐜𝐨𝐮𝐧𝐭(𝒘, 𝒄) + 𝑽
              
Where:
𝑽 represents the vocabulary of the training set.

vocabulary

In [ ]:
from collections import Counter

vocab = set()
for doc in data:
    vocab.update(doc)
V = len(vocab)


P(c)

In [ ]:
from collections import defaultdict
import math

class_docs = defaultdict(list)
for tokens, label in zip(data, labels):
    class_docs[label].append(tokens)

N = len(data)
P_c = {c: len(docs)/N for c, docs in class_docs.items()}


p(w\c)

In [ ]:
word_counts = {}
total_words = {}

for c, docs in class_docs.items():
    word_counts[c] = Counter([w for doc in docs for w in doc])
    total_words[c] = sum(word_counts[c].values())

def P_w_given_c(word, c):
    return (word_counts[c][word] + 1) / (total_words[c] + V)


classify a new article

In [ ]:
def classify(tokens):
    scores = {}
    for c in P_c:
        # log probabilities to avoid underflow
        log_prob = math.log(P_c[c])
        for w in tokens:
            if w in vocab:
                log_prob += math.log(P_w_given_c(w, c))
        scores[c] = log_prob
    return max(scores, key=scores.get), scores


# Testing

In [ ]:
# import os
# import shutil

base_path = "All-in-many_classification_testing/All-in-many_classification_testing/"
articles_path = os.path.join(base_path, "articles")
labels_path = os.path.join(base_path, "labels")
# tokens_path = os.path.joint(base_path,"tokens")

# # Create folders if they don't exist
# os.makedirs(articles_path, exist_ok=True)
# os.makedirs(labels_path, exist_ok=True)
# os.makedirs(tokens_path, exist_ok=True)

# # Iterate over files
# for filename in os.listdir(base_path):
#     filepath = os.path.join(base_path, filename)

#     # Skip directories (articles/ and labels/)
#     if os.path.isdir(filepath):
#         continue

#     # Classification rule
#     if "label" in filename.lower():         # if the filename contains "label"
#         shutil.move(filepath, labels_path)  # move to labels/
#     if "test" in filename.lower():
#         shutil.move(filepath, tokens_path)  # move to articles/
#     else:
#         shutil.move(filepath,articles_path)

# print("✔ Files have been separated into 'articles/' and 'labels/' folders.")


In [ ]:
### TESTING ###

def test_document(filepath):
    with open(filepath, encoding="utf-8") as f:
        text = f.read()
    tokens = preprocess_text(text)
    return classify(tokens)

import os
import math

def read_file_safe(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except UnicodeDecodeError:
        with open(path, encoding="latin-1") as f:
            return f.read()


def evaluate_test_folder(test_articles_path, test_labels_path):
    true_labels = []
    predicted_labels = []

    # Collect article files
    article_files = sorted(os.listdir(test_articles_path))

    for article_file in article_files:
        article_path = os.path.join(test_articles_path, article_file)
        prefix = article_file.split(".")[0]
        expected_label_file = prefix + "_label.txt"
        label_path = os.path.join(test_labels_path, expected_label_file)

        if not os.path.isfile(label_path):
            print(f"⚠️ Missing label for {article_file}")
            continue

    tokens = preprocess_text(read_file_safe(article_path))
    true_label = read_file_safe(label_path).strip()
    pred_label, _ = classify(tokens)

    true_labels.append(true_label)
    predicted_labels.append(pred_label)


    # for article_file in article_files:
    #     article_path = os.path.join(test_articles_path, article_file)

    #     # Extract prefix (remove extension if any)
    #     prefix = article_file.split(".")[0]

    #     # Find matching label file
    #     label_candidates = [
    #         f for f in os.listdir(test_labels_path)
    #         if f.startswith(prefix + "_label")
    #     ]

    #     if not label_candidates:
    #         print(f"⚠️ No label found for: {article_file}")
    #         continue

    #     label_file = label_candidates[0]
    #     label_path = os.path.join(test_labels_path, label_file)

    #     # Read article
    #     with open(article_path, "r", encoding="utf-8", errors="ignore") as f:
    #         text = f.read()
    #     tokens = preprocess_text(text)

    #     # Read label
    #     with open(label_path, "r", encoding="utf-8", errors="ignore") as f:
    #         true_label = f.read().strip()

    #     # Predict
    #     pred_label, _ = classify(tokens)

    #     true_labels.append(true_label)
    #     predicted_labels.append(pred_label)

    # Accuracy
    if len(true_labels) == 0:
        print("❌ No test items classified. Check filenames.")
        return

    accuracy = sum(1 for y,p in zip(true_labels, predicted_labels) if y == p) / len(true_labels)
    print(f"✅ Test accuracy = {accuracy:.4f}")

    return true_labels, predicted_labels


# Example usage:
evaluate_test_folder(articles_path, labels_path)


✅ Test accuracy = 0.5122


(['3',
  '2',
  '2',
  '1',
  '2',
  '2',
  '4',
  '3',
  '4',
  '1',
  '4',
  '1',
  '1',
  '1',
  '4',
  '2',
  '3',
  '4',
  '2',
  '1',
  '1',
  '2',
  '2',
  '2',
  '4',
  '1',
  '2',
  '4',
  '4',
  '4',
  '4',
  '4',
  '2',
  '1',
  '2',
  '2',
  '2',
  '2',
  '2',
  '2',
  '1'],
 ['1',
  '3',
  '2',
  '3',
  '2',
  '2',
  '2',
  '3',
  '4',
  '2',
  '2',
  '1',
  '1',
  '2',
  '1',
  '3',
  '3',
  '2',
  '2',
  '1',
  '3',
  '2',
  '1',
  '3',
  '2',
  '1',
  '2',
  '2',
  '2',
  '2',
  '1',
  '3',
  '2',
  '1',
  '2',
  '2',
  '2',
  '3',
  '2',
  '2',
  '1'])

# Interface

In [ ]:
# Map labels to class names
num_to_label = {
    "1": "Metaheuristics",
    "2": "Machine & Deep Learning",
    "3": "Combination of Metaheuristics & Machine/Deep Learning",
    "4": "Others"
}


labels_summary = []
for i, lbl_num in enumerate(labels, start=1):  # article numbers start from 1
    labels_summary.append({
        "article_num": i,
        "class_num": lbl_num,
        "label": num_to_label.get(lbl_num, "Unknown")  # map number to text
    })


In [ ]:
import tkinter as tk
from tkinter import ttk
from collections import Counter

# --- Map numeric labels to text labels ---
num_to_label = {
    "1": "Metaheuristics",
    "2": "Machine & Deep Learning",
    "3": "Combination of Metaheuristics & Machine/Deep Learning",
    "4": "Others"
}

# -------------------------
# Prepare per-article summary
labels_summary = []
for i, lbl_num in enumerate(labels, start=1):
    labels_summary.append({
        "article_num": i,
        "class_num": lbl_num,
        "label": num_to_label.get(lbl_num, "Unknown")
    })

# -------------------------
# Probabilities
label_counts = Counter(labels)
total_docs = len(labels)
class_probs = {lbl: label_counts[lbl]/total_docs for lbl in label_counts}

# Conditional probabilities
vocab = set(token for doc in data for token in doc)
token_counts = {lbl: Counter() for lbl in label_counts}
total_tokens_per_class = {lbl: 0 for lbl in label_counts}

for doc, lbl in zip(data, labels):
    token_counts[lbl].update(doc)
    total_tokens_per_class[lbl] += len(doc)

V = len(vocab)
conditional_probs = []
for token in vocab:
    row = {"token": token}
    for lbl in label_counts:
        row[lbl] = (token_counts[lbl][token] + 1) / (total_tokens_per_class[lbl] + V)
    conditional_probs.append(row)

# -------------------------
# Tkinter UI
root = tk.Tk()
root.title("Naive Bayes Visualization & Training")
root.geometry("1000x700")
root.configure(bg="#f5f5f5")

button_frame = tk.Frame(root, bg="#f5f5f5")
button_frame.pack(pady=20)
output_frame = tk.Frame(root, bg="#f5f5f5")
output_frame.pack(pady=10, fill="both", expand=True)

# Article dropdown
article_var = tk.StringVar()
article_selector = ttk.Combobox(button_frame, textvariable=article_var)
article_selector['values'] = [f"Article {i}" for i in range(1, len(data)+1)]
article_selector.current(0)
article_selector.grid(row=0, column=0, padx=20)

# Buttons
btn_visualize = ttk.Button(button_frame, text="Visualization")
btn_visualize.grid(row=0, column=1, padx=20)
btn_train = ttk.Button(button_frame, text="Training")
btn_train.grid(row=0, column=2, padx=20)
btn_test = ttk.Button(button_frame, text="Testing")
btn_test.grid(row=0, column=3, padx=20)


# -------------------------
# Functions
def clear_output():
    for widget in output_frame.winfo_children():
        widget.destroy()

def show_visualization():
    clear_output()
    ttk.Label(output_frame, text="📊 Data Summary", font=("Arial", 14, "bold")).pack(pady=5)
    
    tree = ttk.Treeview(output_frame, columns=("article_num", "class_num", "label"), show="headings", height=15)
    tree.pack(padx=10, pady=10, fill="both", expand=True)

    tree.heading("article_num", text="Article Number")
    tree.heading("class_num", text="Class Number")
    tree.heading("label", text="Label")

    for row in labels_summary:
        tree.insert("", "end", values=(row["article_num"], row["class_num"], row["label"]))

def show_training(article_index=0):
    clear_output()

    # --- Table 1: Class probabilities ---
    ttk.Label(output_frame, text="🧠 Estimated Class Probabilities", font=("Arial", 14, "bold")).pack(pady=5)
    
    cols = [f'{lbl} ({num_to_label[lbl]})' for lbl in sorted(class_probs.keys())]
    tree1 = ttk.Treeview(output_frame, columns=cols, show="headings", height=3)
    tree1.pack(padx=10, pady=10, fill="x")

    for col in cols:
        tree1.heading(col, text=col)

    tree1.insert("", "end", values=[f"{class_probs[lbl]:.4f}" for lbl in sorted(class_probs.keys())])

    # --- Table 2: Conditional probabilities for selected article ---
    ttk.Label(output_frame, text=f"📈 Conditional Probabilities P(w|c) for Article {article_index+1}", font=("Arial", 14, "bold")).pack(pady=5)
    
    tree2 = ttk.Treeview(output_frame, columns=["Token"] + sorted(class_probs.keys()), show="headings", height=20)
    tree2.pack(padx=10, pady=10, fill="both", expand=True)

    tree2.heading("Token", text="Token")
    for lbl in sorted(class_probs.keys()):
        tree2.heading(lbl, text=f"{lbl} ({num_to_label[lbl]})")

    tokens_in_article = set(data[article_index])

    # Filter only tokens in the selected article
    article_tokens = [row for row in conditional_probs if row["token"] in tokens_in_article]

    # Sort alphabetically by token
    article_tokens_sorted = sorted(article_tokens, key=lambda x: x["token"])

    for row in article_tokens_sorted:
        tree2.insert("", "end", values=[row["token"]] + [f"{row[lbl]:.4f}" for lbl in sorted(class_probs.keys())])


def show_testing(test_articles_path, test_labels_path):
    clear_output()
    ttk.Label(output_frame, text="🧪 Testing Results", font=("Arial", 14, "bold")).pack(pady=5)

    # Prepare Treeview
    tree = ttk.Treeview(output_frame, columns=("article_num", "true_label", "predicted_label"), show="headings", height=20)
    tree.pack(padx=10, pady=10, fill="both", expand=True)

    tree.heading("article_num", text="Article Number")
    tree.heading("true_label", text="True Label")
    tree.heading("predicted_label", text="Predicted Label")

    # Collect test files
    article_files = sorted(os.listdir(test_articles_path))

    for i, article_file in enumerate(article_files, start=1):
        article_path = os.path.join(test_articles_path, article_file)
        prefix = article_file.split(".")[0]

        # Find matching label
        label_candidates = [
            f for f in os.listdir(test_labels_path)
            if f.startswith(prefix + "_label")
        ]
        if not label_candidates:
            continue
        label_file = label_candidates[0]
        label_path = os.path.join(test_labels_path, label_file)

        # Read article
        with open(article_path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        tokens = preprocess_text(text)

        # Read true label
        with open(label_path, "r", encoding="utf-8", errors="ignore") as f:
            true_label = f.read().strip()

        # Predict
        pred_label, _ = classify(tokens)

        # Insert in table
        tree.insert("", "end", values=(i, true_label, pred_label))



# -------------------------
# Bind dropdown change
def on_article_change(event):
    idx = int(article_var.get().split()[1]) - 1
    show_training(idx)

article_selector.bind("<<ComboboxSelected>>", on_article_change)

# Replace these paths with your real testing folders
test_articles_path = "All-in-many_classification_testing/All-in-many_classification_testing/articles/"
test_labels_path = "All-in-many_classification_testing/All-in-many_classification_testing/labels/"

btn_test.config(command=lambda: show_testing(test_articles_path, test_labels_path))


# Initially show first article
show_training(0)
btn_visualize.config(command=show_visualization)
btn_train.config(command=lambda: show_training(int(article_var.get().split()[1])-1))

root.mainloop()
